# Self-RAG: Self-Reflective Retrieval-Augmented Generation

## Overview

**Self-RAG** (Asai et al., 2023) is a framework that teaches a language model to
**reflect on its own generation process** through special *reflection tokens*. Unlike
standard RAG — which always retrieves and blindly trusts the context — Self-RAG gives the
model agency to:

1. **Decide** whether retrieval is even necessary for a given query.
2. **Judge** whether each retrieved passage is relevant.
3. **Verify** whether the generated response is actually supported by the evidence.
4. **Rate** the overall usefulness of the response.

These four reflection steps dramatically improve factual accuracy and reduce hallucination,
at the cost of additional LLM calls per query.

## Motivation

Standard RAG pipelines have a fundamental design flaw: they **always retrieve**, regardless
of whether the model already knows the answer, and they **never verify** whether the
retrieved context actually supports the generated response. This leads to:

- **Unnecessary retrieval** for simple / well-known questions (wasted latency & compute).
- **Hallucination** when retrieved passages are off-topic but the model plows ahead anyway.
- **Unsupported claims** that sound authoritative but aren't grounded in the evidence.

Self-RAG addresses all three issues by inserting explicit reflection checkpoints into the
generation loop.

## Key Components

| Component | Reflection Token | Purpose |
|---|---|---|
| Retrieval Decision | `[Retrieve]` | Should I retrieve external knowledge? |
| Relevance Check | `[Relevant]` | Is this passage relevant to the query? |
| Support Verification | `[Supported]` | Is the answer supported by evidence? |
| Utility Rating | `[Useful]` | How useful is this response overall? |

## Method Details

1. **Retrieval Decision** — The LLM evaluates the query and decides if external knowledge
   is needed. Factual questions about obscure topics → yes. Simple greetings → no.
2. **Document Retrieval** — If retrieval is triggered, top-k passages are fetched from FAISS.
3. **Relevance Evaluation** — Each passage is independently scored for relevance. Irrelevant
   ones are discarded before generation.
4. **Response Generation** — The LLM generates an answer using only the relevant passages.
5. **Support Assessment** — The LLM checks if the generated answer is actually supported by
   the evidence (fully / partially / not at all).
6. **Utility Rating** — The LLM rates the final answer on a 1–5 usefulness scale.
7. **Adaptive Loop** — If support is weak or utility is low, the system can re-retrieve
   with a reformulated query or regenerate with different context.

<div style="text-align: center;">

<img src="../images/self_rag.svg" alt="Self-RAG" style="width:100%; height:auto;">

</div>

## 1 — Why Standard RAG Falls Short

Before building Self-RAG, let's understand **exactly what's wrong** with the naive
retrieve-then-generate pipeline.

### Problem 1: Blind Retrieval
Standard RAG retrieves for *every* query, even:
- "What is 2 + 2?" — The model knows this; retrieval adds latency for nothing.
- "Write me a poem about cats." — Creative tasks don't benefit from document lookup.

### Problem 2: No Relevance Filtering
If the top-k retrieved passages happen to be about a different topic (e.g., the query is
about *climate change* but the closest vector is about *weather forecasting*), the model
may incorporate irrelevant information and hallucinate.

### Problem 3: No Verification
Even when the context is relevant, the model might generate claims that **go beyond** what
the evidence actually says. Standard RAG has no mechanism to catch this.

### Problem 4: No Quality Signal
There is no feedback loop. The pipeline generates one answer and returns it, with no way to
assess whether the answer is actually useful or whether a second attempt might do better.

Self-RAG solves each of these problems with a dedicated reflection step.

## 2 — What Is Self-RAG?

**Self-RAG** (Self-Reflective Retrieval-Augmented Generation) was introduced by
**Asai et al. (2023)** in the paper
*"Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection"*.

### Core Idea
Instead of treating retrieval as a fixed preprocessing step, Self-RAG trains the LLM itself
to emit **special reflection tokens** that control the generation pipeline:

```
Query → [Retrieve] → (optional retrieval) → [Relevant] → filter → Generate
      → [Supported] → verify → [Useful] → rate → Final Answer
```

### Key Innovation
In the original paper, these reflection tokens are **trained into the model's vocabulary**
using a critic model that labels training data. The LLM learns to predict these tokens as
naturally as it predicts the next word.

### Our Approach
Since we can't fine-tune special tokens in a notebook, we **simulate** reflection tokens
using **prompt-based self-evaluation**. We ask the same LLM to play the role of a critic
at each decision point. This captures the *spirit* of Self-RAG while being implementable
with any instruction-following model.

### The Trade-off
Self-RAG requires **multiple LLM calls per query** (typically 4–6 instead of 1–2). This
increases latency by 2–4×. The payoff is significantly higher factual accuracy and the
ability to gracefully handle queries where retrieval would hurt rather than help.

## 3 — The Four Reflection Tokens

Self-RAG's magic comes from four reflection tokens, each corresponding to a distinct
evaluation checkpoint.

### Token 1: `[Retrieve]` — Should I retrieve?

| Value | Meaning |
|---|---|
| **yes** | The query requires external factual knowledge |
| **no** | The model can answer from parametric knowledge alone |
| **continue** | Retrieval was already done; no need to re-retrieve |

**Example decisions:**
- *"What year was the Eiffel Tower built?"* → **yes** (specific factual detail)
- *"Write a haiku about rain."* → **no** (creative task, no facts needed)
- *"Summarize the passage above."* → **continue** (context already present)

### Token 2: `[Relevant]` — Is this passage relevant?

| Value | Meaning |
|---|---|
| **relevant** | The passage contains information pertinent to the query |
| **irrelevant** | The passage is off-topic or not useful for answering |

This is evaluated **per retrieved passage**, so if 5 passages are retrieved, the LLM
makes 5 relevance judgments.

### Token 3: `[Supported]` — Is the answer supported?

| Value | Meaning |
|---|---|
| **fully_supported** | Every claim in the answer is backed by the evidence |
| **partially_supported** | Some claims are supported, others are not |
| **no_support** | The answer contradicts or goes beyond the evidence |

### Token 4: `[Useful]` — How useful is this response?

| Score | Meaning |
|---|---|
| **5** | Excellent — directly and completely answers the question |
| **4** | Good — mostly answers the question with minor gaps |
| **3** | Acceptable — partially addresses the question |
| **2** | Poor — barely relevant to the question |
| **1** | Useless — does not address the question at all |

## 4 — The Self-RAG Loop: Step by Step

Here is the complete Self-RAG decision flow:

```
┌─────────────────────────────────────────────────────────┐
│                     INPUT: Query                         │
└────────────────────────┬────────────────────────────────┘
                         │
                         ▼
              ┌──────────────────┐
              │  [Retrieve] Check │
              │  Need retrieval?  │
              └────────┬─────────┘
                 ┌─────┴──────┐
                 │            │
              yes│            │no
                 ▼            ▼
          ┌──────────┐   Generate from
          │ Retrieve  │   parametric
          │ top-k     │   knowledge only
          │ passages  │       │
          └─────┬────┘       │
                │             │
                ▼             │
     ┌───────────────────┐   │
     │ [Relevant] Check   │   │
     │ per passage        │   │
     └─────────┬─────────┘   │
               │              │
               ▼              │
     Filter to relevant       │
     passages only            │
               │              │
               ▼              ▼
     ┌──────────────────────────┐
     │  Generate response using  │
     │  relevant context (or     │
     │  parametric knowledge)    │
     └────────────┬─────────────┘
                  │
                  ▼
       ┌───────────────────┐
       │ [Supported] Check  │
       │ Evidence support?  │
       └─────────┬─────────┘
                 │
                 ▼
       ┌───────────────────┐
       │  [Useful] Check    │
       │  Quality rating?   │
       └─────────┬─────────┘
            ┌────┴────┐
            │         │
         ≥ 3         < 3
            │         │
            ▼         ▼
        Return    Re-retrieve or
        answer    regenerate
```

The loop can iterate up to a configurable maximum (we use 2 retries) before returning
the best available answer.

## 5 — Environment Setup

In [ ]:
# Install required packages
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

In [ ]:
import os, pathlib

# Google Drive cache for large models
CACHE = "/content/drive/MyDrive/models"
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    CACHE = str(pathlib.Path.home() / ".cache" / "huggingface" / "hub")

os.makedirs(CACHE, exist_ok=True)
os.environ["HF_HOME"]           = CACHE
os.environ["TRANSFORMERS_CACHE"] = CACHE
os.environ["HF_HUB_CACHE"]      = CACHE
print(f"Model cache → {CACHE}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# ── Embedding model ──────────────────────────────────────────────
EMB_ID = "BAAI/bge-base-en-v1.5"
embedder = SentenceTransformer(EMB_ID, cache_folder=CACHE)
print(f"Embedder loaded: {EMB_ID}  dim={embedder.get_sentence_embedding_dimension()}")

# ── LLM: Qwen3-14B  4-bit NF4 ───────────────────────────────────
LLM_ID = "Qwen/Qwen3-14B"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE)
model = AutoModelForCausalLM.from_pretrained(
    LLM_ID, quantization_config=bnb, device_map="auto", cache_dir=CACHE,
)
print(f"LLM loaded: {LLM_ID}  (4-bit NF4)")

In [ ]:
def generate(prompt, max_new_tokens=512, temperature=0.7):
    """Generate a response from Qwen3-14B with /no_think for deterministic output."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": prompt + "\n/no_think"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.9, do_sample=True,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Quick smoke test
print(generate("Say hello in one sentence.", max_new_tokens=30))

## 6 — Synthetic Knowledge Base

We create an inline knowledge base spanning **4 topics** with enough detail to test
retrieval quality and relevance filtering.

| Passages | Topic |
|---|---|
| 1–5 | **Solar Energy & Photovoltaics** |
| 6–10 | **The History of Computing** |
| 11–15 | **Marine Biology & Ocean Ecosystems** |
| 16–20 | **Nutrition & Human Health** |

In [ ]:
KNOWLEDGE_BASE = [
    # ── Solar Energy & Photovoltaics (1-5) ────────────────────────
    "Solar photovoltaic (PV) cells convert sunlight directly into electricity using the "
    "photovoltaic effect. When photons strike the semiconductor material (usually silicon), "
    "they knock electrons loose, creating an electric current. Modern monocrystalline panels "
    "achieve efficiencies of 20-22% in commercial applications.",

    "The levelized cost of solar electricity (LCOE) has dropped by over 89% since 2010, "
    "making it the cheapest source of new electricity generation in most countries. Utility-scale "
    "solar farms now produce power at $20-40 per megawatt-hour, competitive with natural gas "
    "and significantly cheaper than coal.",

    "Solar panel manufacturing involves purifying silicon to 99.9999% purity, slicing it into "
    "thin wafers, doping the wafers with boron and phosphorus to create p-n junctions, and "
    "adding anti-reflective coatings. The energy payback time for a modern panel is 1-2 years, "
    "after which it generates net-positive clean energy for 25-30 more years.",

    "Concentrated solar power (CSP) plants use mirrors or lenses to focus sunlight onto a "
    "receiver that heats a fluid (often molten salt) to drive a steam turbine. Unlike PV, "
    "CSP can store thermal energy for hours, enabling electricity generation after sunset. "
    "The Noor-Ouarzazate complex in Morocco is the world's largest CSP installation.",

    "Perovskite solar cells are an emerging technology that could revolutionize photovoltaics. "
    "These cells use a perovskite-structured compound as the light-absorbing layer and can be "
    "manufactured using simple solution-processing techniques. Lab efficiencies have reached "
    "25.7%, approaching silicon, but long-term stability remains a challenge.",

    # ── History of Computing (6-10) ───────────────────────────────
    "Charles Babbage designed the Analytical Engine in 1837, considered the first general-purpose "
    "computer concept. It featured an arithmetic logic unit, control flow via conditional branching "
    "and loops, and integrated memory — all fundamental components of modern computers. Ada Lovelace "
    "wrote the first algorithm intended for the machine.",

    "The ENIAC (Electronic Numerical Integrator and Computer), completed in 1945, was the first "
    "general-purpose electronic digital computer. It weighed 30 tons, occupied 1,800 square feet, "
    "and used 18,000 vacuum tubes. Despite its size, its computing power was less than a modern "
    "pocket calculator.",

    "The invention of the transistor at Bell Labs in 1947 by Bardeen, Brattain, and Shockley "
    "revolutionized computing. Transistors replaced vacuum tubes, making computers smaller, faster, "
    "cheaper, and more reliable. This breakthrough earned the trio the 1956 Nobel Prize in Physics "
    "and laid the foundation for all modern electronics.",

    "Moore's Law, proposed by Gordon Moore in 1965, observed that the number of transistors on "
    "a microchip doubles approximately every two years while the cost halves. This observation "
    "guided the semiconductor industry for over five decades, driving exponential improvements "
    "in computing performance, storage capacity, and cost efficiency.",

    "The development of the internet began with ARPANET in 1969, initially connecting four "
    "universities. Tim Berners-Lee invented the World Wide Web in 1989 at CERN, creating HTML, "
    "HTTP, and the first web browser. By 2024, over 5.3 billion people worldwide use the internet, "
    "fundamentally transforming communication, commerce, and information access.",

    # ── Marine Biology & Ocean Ecosystems (11-15) ─────────────────
    "Coral reefs occupy less than 1% of the ocean floor yet support approximately 25% of all "
    "marine species. They are built by colonies of tiny coral polyps that secrete calcium carbonate "
    "skeletons. The symbiotic relationship between coral and zooxanthellae algae provides up to "
    "90% of the coral's energy through photosynthesis.",

    "The deep ocean (below 200 meters) is the largest habitat on Earth but remains largely "
    "unexplored. Organisms here survive extreme pressure, near-freezing temperatures, and total "
    "darkness. Hydrothermal vents support unique ecosystems based on chemosynthesis rather than "
    "photosynthesis, hosting giant tube worms, eyeless shrimp, and specialized bacteria.",

    "Ocean acidification, caused by absorption of excess atmospheric CO₂, threatens marine "
    "organisms that build calcium carbonate shells or skeletons. Since the Industrial Revolution, "
    "ocean pH has dropped by 0.1 units (a 26% increase in acidity). Pteropods, oysters, and "
    "corals are particularly vulnerable to these changes.",

    "The Great Barrier Reef, stretching over 2,300 kilometers along Australia's northeast coast, "
    "is the world's largest coral reef system. It comprises over 2,900 individual reefs and "
    "supports more than 1,500 species of fish, 400 types of coral, and 4,000 species of mollusk. "
    "Rising ocean temperatures have caused severe mass bleaching events in 2016, 2017, 2020, and 2022.",

    "Marine protected areas (MPAs) are designated zones where human activity is restricted to "
    "conserve biodiversity and ecosystem services. Studies show that well-enforced MPAs increase "
    "fish biomass by an average of 670% compared to unprotected areas. Currently, about 8% of "
    "the global ocean is covered by MPAs, though the target is 30% by 2030.",

    # ── Nutrition & Human Health (16-20) ──────────────────────────
    "Macronutrients — carbohydrates, proteins, and fats — provide the body with energy measured "
    "in calories. Carbohydrates yield 4 kcal/g, proteins 4 kcal/g, and fats 9 kcal/g. The "
    "recommended macronutrient distribution for adults is 45-65% carbs, 10-35% protein, and "
    "20-35% fat of total daily caloric intake.",

    "Vitamin D is synthesized in the skin upon exposure to UVB radiation and plays a critical "
    "role in calcium absorption, bone health, and immune function. Deficiency affects an estimated "
    "1 billion people worldwide and is associated with increased risk of osteoporosis, autoimmune "
    "diseases, cardiovascular disease, and certain cancers.",

    "The gut microbiome consists of trillions of microorganisms (bacteria, fungi, viruses) living "
    "in the digestive tract. These microbes aid digestion, produce vitamins (B12, K), regulate "
    "the immune system, and communicate with the brain via the gut-brain axis. Diet diversity "
    "is the strongest predictor of microbiome diversity and health.",

    "Omega-3 fatty acids (EPA and DHA), found primarily in fatty fish like salmon, mackerel, and "
    "sardines, have potent anti-inflammatory properties. Clinical trials show they reduce "
    "triglyceride levels by 15-30%, lower blood pressure, and may reduce the risk of heart attack "
    "and stroke. The recommended intake is at least 250-500 mg combined EPA and DHA per day.",

    "Intermittent fasting (IF) is an eating pattern that cycles between periods of fasting and "
    "eating. The most studied protocols are 16:8 (16 hours fasting, 8 hours eating) and 5:2 "
    "(5 normal days, 2 low-calorie days per week). Research shows IF can improve insulin "
    "sensitivity, promote autophagy (cellular cleanup), and support weight management.",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} passages")
for i, p in enumerate(KNOWLEDGE_BASE):
    print(f"  [{i+1:>2}] {p[:80]}...")

## 7 — Building the FAISS Vector Store

We encode all 20 passages with `bge-base-en-v1.5` and store them in a flat FAISS index
for similarity search.

In [ ]:
import numpy as np
import faiss

# Encode all passages
embeddings = embedder.encode(KNOWLEDGE_BASE, normalize_embeddings=True, show_progress_bar=True)
embeddings = np.array(embeddings, dtype="float32")

# Build FAISS index (Inner Product = cosine similarity on normalized vectors)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dimension={dimension}")

def retrieve(query, top_k=5):
    """Retrieve top-k most similar passages from the knowledge base."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "passage": KNOWLEDGE_BASE[idx],
            "score": float(score),
            "index": int(idx),
        })
    return results

# Quick test
test_results = retrieve("How do solar panels work?", top_k=3)
for r in test_results:
    print(f"  [{r['index']+1:>2}] score={r['score']:.4f}  {r['passage'][:70]}...")
print("\nRetrieval function ready ✓")

## 8 — Standard RAG Baseline

Before implementing Self-RAG, let's build a **standard RAG pipeline** that always retrieves
and generates without any reflection. This gives us a baseline to compare against.

In [ ]:
def standard_rag(query, top_k=3):
    """Standard RAG: always retrieve, then generate. No reflection."""
    # Step 1: Always retrieve
    results = retrieve(query, top_k=top_k)

    # Step 2: Build context from ALL retrieved passages (no filtering)
    context = "\n\n".join(f"[Passage {i+1}]: {r['passage']}" for i, r in enumerate(results))

    # Step 3: Generate response
    prompt = f"""Answer the following question using ONLY the provided context.
If the context doesn't contain enough information, say so.

Context:
{context}

Question: {query}

Answer:"""

    answer = generate(prompt, max_new_tokens=300)
    return {
        "query": query,
        "answer": answer,
        "passages_used": len(results),
        "retrieval_scores": [r["score"] for r in results],
    }

# Test on a straightforward question
result = standard_rag("What is the efficiency of modern solar panels?")
print(f"Query: {result['query']}")
print(f"Passages used: {result['passages_used']}")
print(f"Retrieval scores: {[f'{s:.4f}' for s in result['retrieval_scores']]}")
print(f"Answer: {result['answer']}")

### Demonstrating Standard RAG Limitations

Let's try a query where the model **should not retrieve** — a simple arithmetic question.
Standard RAG will retrieve anyway, wasting time and potentially confusing the answer.

In [ ]:
# Standard RAG on a question that doesn't need retrieval
result = standard_rag("What is 15 multiplied by 7?")
print(f"Query: {result['query']}")
print(f"Passages retrieved: {result['passages_used']}  (unnecessary!)")
print(f"Scores: {[f'{s:.4f}' for s in result['retrieval_scores']]}")
print(f"Answer: {result['answer']}")
print()
print("⚠ Standard RAG retrieved 3 passages for a simple math question!")
print("  This adds latency and the irrelevant context might confuse the model.")

## 9 — Implementing Self-RAG from Scratch

Since we can't train special reflection tokens into the model's vocabulary, we **simulate**
them using prompt-based self-evaluation. Each reflection step becomes a structured prompt
that asks the LLM to make a judgment call.

We'll build each reflection function independently, test it, and then combine them into
the full Self-RAG loop.

### Step 1: `[Retrieve]` — Retrieval Necessity Check

The first reflection asks: *"Does this question require external knowledge, or can I
answer it from what I already know?"*

This is the most impactful reflection — it prevents unnecessary retrieval entirely.

In [ ]:
def check_retrieval_need(query):
    """
    [Retrieve] reflection token: Decide whether retrieval is necessary.
    Returns: 'yes', 'no', or 'continue'
    """
    prompt = f"""You are a retrieval decision module. Given a user query, decide whether
external document retrieval is necessary to answer it accurately.

Rules:
- Answer "yes" if the query asks for specific facts, data, technical details, or
  domain-specific knowledge that requires authoritative sources.
- Answer "no" if the query is about common knowledge, simple calculations, creative
  tasks, opinions, or conversational exchanges.
- Answer "continue" if sufficient context has already been provided.

Respond with EXACTLY one word: yes, no, or continue

Query: {query}

Decision:"""

    response = generate(prompt, max_new_tokens=10, temperature=0.1)
    # Parse the response — extract the first valid token
    response_lower = response.strip().lower()
    for token in ["yes", "no", "continue"]:
        if token in response_lower:
            return token
    return "yes"  # Default to retrieval if uncertain

# Test on various query types
test_queries = [
    "What is the efficiency of perovskite solar cells?",
    "What is 25 times 4?",
    "Write a short poem about the ocean.",
    "How does ocean acidification affect marine organisms?",
    "What is the capital of France?",
]

print("[Retrieve] Reflection Token Tests:")
print("=" * 60)
for q in test_queries:
    decision = check_retrieval_need(q)
    print(f"  [{decision:>8}]  {q}")

### Step 2: `[Relevant]` — Passage Relevance Check

For each retrieved passage, we ask the LLM: *"Is this passage relevant to answering the
query?"*

This filters out noise before generation, preventing the model from being misled by
off-topic context.

In [ ]:
def check_relevance(query, passage):
    """
    [Relevant] reflection token: Evaluate if a passage is relevant to the query.
    Returns: 'relevant' or 'irrelevant'
    """
    prompt = f"""You are a relevance evaluation module. Given a query and a retrieved passage,
determine whether the passage contains information that is relevant and useful for
answering the query.

A passage is "relevant" if it contains facts, data, or explanations that directly help
answer the query. A passage is "irrelevant" if it discusses a different topic or provides
no useful information for the query.

Respond with EXACTLY one word: relevant or irrelevant

Query: {query}
Passage: {passage}

Relevance:"""

    response = generate(prompt, max_new_tokens=10, temperature=0.1)
    response_lower = response.strip().lower()
    if "irrelevant" in response_lower:
        return "irrelevant"
    if "relevant" in response_lower:
        return "relevant"
    return "relevant"  # Default to relevant if uncertain

# Test: retrieve passages for a marine biology question and check relevance
query = "How does ocean acidification affect coral reefs?"
passages = retrieve(query, top_k=5)
print(f"Query: {query}")
print(f"{'='*70}")
for r in passages:
    relevance = check_relevance(query, r["passage"])
    marker = "✓" if relevance == "relevant" else "✗"
    print(f"  [{marker} {relevance:>10}]  (score={r['score']:.4f})  {r['passage'][:65]}...")

### Step 3: `[Supported]` — Evidence Support Verification

After generating an answer, we check: *"Is this answer actually supported by the evidence
I used?"*

This is the **hallucination detector** — it catches cases where the model generates
plausible-sounding claims that aren't grounded in the retrieved passages.

In [ ]:
def check_support(query, answer, passages):
    """
    [Supported] reflection token: Check if the answer is supported by evidence.
    Returns: 'fully_supported', 'partially_supported', or 'no_support'
    """
    context = "\n".join(f"- {p}" for p in passages)

    prompt = f"""You are a support verification module. Given a query, a generated answer,
and the evidence passages used to generate it, determine how well the answer is
supported by the evidence.

Criteria:
- "fully_supported": Every factual claim in the answer can be traced back to the evidence.
- "partially_supported": Some claims are supported but others go beyond the evidence.
- "no_support": The answer contradicts the evidence or makes claims not found in it.

Respond with EXACTLY one phrase: fully_supported, partially_supported, or no_support

Query: {query}

Evidence:
{context}

Answer: {answer}

Support level:"""

    response = generate(prompt, max_new_tokens=15, temperature=0.1)
    response_lower = response.strip().lower().replace(" ", "_")
    for level in ["fully_supported", "partially_supported", "no_support"]:
        if level in response_lower:
            return level
    # Check partial matches
    if "fully" in response_lower:
        return "fully_supported"
    if "partial" in response_lower:
        return "partially_supported"
    return "no_support"

# Test: generate an answer and check support
query = "What is the energy payback time for a modern solar panel?"
passages = retrieve(query, top_k=3)
relevant_texts = [r["passage"] for r in passages]
context = "\n\n".join(relevant_texts)

answer = generate(f"""Answer this question using the provided context.

Context:
{context}

Question: {query}

Answer:""", max_new_tokens=200)

support = check_support(query, answer, relevant_texts)
print(f"Query:   {query}")
print(f"Answer:  {answer}")
print(f"Support: {support}")

### Step 4: `[Useful]` — Utility Rating

Finally, we rate the overall usefulness of the response on a 1–5 scale. Low scores
trigger re-retrieval or regeneration.

In [ ]:
def check_utility(query, answer):
    """
    [Useful] reflection token: Rate the usefulness of the answer (1-5).
    Returns: integer score 1-5
    """
    prompt = f"""You are a utility evaluation module. Rate how useful the following answer
is for the given query on a scale of 1 to 5.

Rating scale:
5 = Excellent: directly and completely answers the question with accurate details
4 = Good: mostly answers the question with minor gaps
3 = Acceptable: partially addresses the question
2 = Poor: barely relevant or mostly unhelpful
1 = Useless: does not address the question at all

Respond with EXACTLY one number: 1, 2, 3, 4, or 5

Query: {query}
Answer: {answer}

Utility score:"""

    response = generate(prompt, max_new_tokens=10, temperature=0.1)
    # Extract the first digit found
    for char in response.strip():
        if char.isdigit() and char in "12345":
            return int(char)
    return 3  # Default to middle if parsing fails

# Test utility scoring
test_cases = [
    ("What is Moore's Law?", "Moore's Law states that the number of transistors on a microchip "
     "doubles approximately every two years while the cost halves. It was proposed by Gordon "
     "Moore in 1965 and has guided the semiconductor industry for over five decades."),
    ("What is Moore's Law?", "I don't know."),
    ("What is Moore's Law?", "The ocean is very deep and contains many fish species."),
]

print("Utility Score Tests:")
print("=" * 60)
for query, answer in test_cases:
    score = check_utility(query, answer)
    print(f"  Score={score}/5  Q: {query}")
    print(f"           A: {answer[:70]}...")
    print()

## 10 — Complete Self-RAG Pipeline

Now we combine all four reflection tokens into a single pipeline with an adaptive retry
loop. If the support is weak or the utility is low, the system re-retrieves with an
expanded search or regenerates with different context.

In [ ]:
import time

def self_rag(query, top_k=5, utility_threshold=3, max_retries=2):
    """
    Complete Self-RAG pipeline with all four reflection tokens.

    Args:
        query: The user's question
        top_k: Number of passages to retrieve
        utility_threshold: Minimum utility score to accept (1-5)
        max_retries: Maximum number of retry attempts

    Returns:
        Dictionary with answer, reflection trace, and metadata
    """
    trace = []         # Log of all reflection decisions
    start = time.time()

    for attempt in range(max_retries + 1):
        step_prefix = f"[Attempt {attempt+1}]"

        # ── Step 1: [Retrieve] — Should we retrieve? ────────────
        retrieve_decision = check_retrieval_need(query)
        trace.append(f"{step_prefix} [Retrieve] → {retrieve_decision}")

        relevant_passages = []

        if retrieve_decision == "yes":
            # ── Step 2: Retrieve passages ────────────────────────
            k = top_k + (attempt * 2)  # Expand search on retries
            retrieved = retrieve(query, top_k=min(k, 10))
            trace.append(f"{step_prefix} Retrieved {len(retrieved)} passages (top_k={k})")

            # ── Step 3: [Relevant] — Filter passages ─────────────
            for r in retrieved:
                relevance = check_relevance(query, r["passage"])
                marker = "✓" if relevance == "relevant" else "✗"
                trace.append(f"{step_prefix} [Relevant] {marker} (score={r['score']:.3f}) "
                             f"{r['passage'][:50]}...")
                if relevance == "relevant":
                    relevant_passages.append(r["passage"])

            trace.append(f"{step_prefix} → {len(relevant_passages)}/{len(retrieved)} "
                         f"passages deemed relevant")

        # ── Step 4: Generate response ────────────────────────────
        if relevant_passages:
            context = "\n\n".join(f"[Evidence {i+1}]: {p}"
                                   for i, p in enumerate(relevant_passages))
            prompt = f"""Answer the following question using ONLY the provided evidence.
Be specific and cite evidence where possible.

Evidence:
{context}

Question: {query}

Answer:"""
        else:
            trace.append(f"{step_prefix} No retrieval / no relevant passages → "
                         f"generating from parametric knowledge")
            prompt = f"""Answer the following question using your own knowledge.
Be concise and accurate.

Question: {query}

Answer:"""

        answer = generate(prompt, max_new_tokens=300)
        trace.append(f"{step_prefix} Generated answer ({len(answer)} chars)")

        # ── Step 5: [Supported] — Check evidence support ────────
        if relevant_passages:
            support = check_support(query, answer, relevant_passages)
        else:
            support = "no_evidence"  # Can't check support without evidence
        trace.append(f"{step_prefix} [Supported] → {support}")

        # ── Step 6: [Useful] — Rate utility ──────────────────────
        utility = check_utility(query, answer)
        trace.append(f"{step_prefix} [Useful] → {utility}/5")

        # ── Step 7: Accept or retry ──────────────────────────────
        if utility >= utility_threshold and support != "no_support":
            trace.append(f"{step_prefix} ✓ Answer accepted (utility={utility}, "
                         f"support={support})")
            break
        elif attempt < max_retries:
            trace.append(f"{step_prefix} ✗ Answer rejected — retrying with expanded search")
        else:
            trace.append(f"{step_prefix} ⚠ Max retries reached — returning best effort")

    elapsed = time.time() - start
    return {
        "query":              query,
        "answer":             answer,
        "retrieve_decision":  retrieve_decision,
        "relevant_passages":  len(relevant_passages),
        "support":            support,
        "utility":            utility,
        "attempts":           attempt + 1,
        "elapsed_seconds":    round(elapsed, 2),
        "trace":              trace,
    }

print("Self-RAG pipeline defined ✓")

## 11 — Self-RAG in Action: Example Queries

### Example 1: Factual Query (Retrieval Expected)

A domain-specific question about solar energy — the model should retrieve, find relevant
passages, and generate a well-supported answer.

In [ ]:
result = self_rag("How do perovskite solar cells compare to silicon panels in efficiency?")

print(f"Query:    {result['query']}")
print(f"Answer:   {result['answer']}")
print(f"{'─'*70}")
print(f"[Retrieve]:  {result['retrieve_decision']}")
print(f"[Relevant]:  {result['relevant_passages']} passages kept")
print(f"[Supported]: {result['support']}")
print(f"[Useful]:    {result['utility']}/5")
print(f"Attempts:    {result['attempts']}")
print(f"Time:        {result['elapsed_seconds']}s")
print(f"\n{'─'*70}")
print("Full Reflection Trace:")
for step in result["trace"]:
    print(f"  {step}")

### Example 2: Simple Query (Retrieval Should Be Skipped)

A basic arithmetic question — Self-RAG should decide **not** to retrieve and answer
directly from parametric knowledge.

In [ ]:
result = self_rag("What is 42 divided by 6?")

print(f"Query:    {result['query']}")
print(f"Answer:   {result['answer']}")
print(f"{'─'*70}")
print(f"[Retrieve]:  {result['retrieve_decision']}  ← should be 'no'")
print(f"[Relevant]:  {result['relevant_passages']} passages (none retrieved)")
print(f"[Supported]: {result['support']}")
print(f"[Useful]:    {result['utility']}/5")
print(f"Time:        {result['elapsed_seconds']}s")
print(f"\n{'─'*70}")
print("Full Reflection Trace:")
for step in result["trace"]:
    print(f"  {step}")
print()
print("💡 Self-RAG correctly skipped retrieval, saving time and avoiding")
print("   the confusion that irrelevant passages would introduce.")

### Example 3: Cross-Domain Query (Tests Relevance Filtering)

A question about marine biology — some retrieved passages from other domains (solar,
computing) should be filtered out by the `[Relevant]` check.

In [ ]:
result = self_rag("What is the relationship between coral reefs and zooxanthellae algae?")

print(f"Query:    {result['query']}")
print(f"Answer:   {result['answer']}")
print(f"{'─'*70}")
print(f"[Retrieve]:  {result['retrieve_decision']}")
print(f"[Relevant]:  {result['relevant_passages']} passages kept (out of retrieved)")
print(f"[Supported]: {result['support']}")
print(f"[Useful]:    {result['utility']}/5")
print(f"Attempts:    {result['attempts']}")
print(f"Time:        {result['elapsed_seconds']}s")
print(f"\n{'─'*70}")
print("Full Reflection Trace:")
for step in result["trace"]:
    print(f"  {step}")

### Example 4: Nutrition Query (Tests Full Pipeline)

A question about omega-3 fatty acids — should retrieve from the nutrition cluster and
verify the answer against evidence.

In [ ]:
result = self_rag("What are the health benefits of omega-3 fatty acids and how much should I consume daily?")

print(f"Query:    {result['query']}")
print(f"Answer:   {result['answer']}")
print(f"{'─'*70}")
print(f"[Retrieve]:  {result['retrieve_decision']}")
print(f"[Relevant]:  {result['relevant_passages']} passages kept")
print(f"[Supported]: {result['support']}")
print(f"[Useful]:    {result['utility']}/5")
print(f"Attempts:    {result['attempts']}")
print(f"Time:        {result['elapsed_seconds']}s")
print(f"\nReflection Trace:")
for step in result["trace"]:
    print(f"  {step}")

## 12 — Head-to-Head: Standard RAG vs Self-RAG

Let's run the **same queries** through both pipelines and compare results.
We'll examine:
1. Whether retrieval was skipped when appropriate
2. How many passages were actually used
3. Answer quality (support level and utility score)
4. Latency difference

In [ ]:
comparison_queries = [
    "What is the efficiency of perovskite solar cells?",
    "What is 15 multiplied by 7?",
    "How do marine protected areas affect fish populations?",
    "Write a short haiku about the moon.",
]

print(f"{'='*80}")
print(f"{'STANDARD RAG vs SELF-RAG COMPARISON':^80}")
print(f"{'='*80}")

for query in comparison_queries:
    print(f"\nQuery: {query}")
    print(f"{'─'*80}")

    # Standard RAG
    t0 = time.time()
    std_result = standard_rag(query, top_k=3)
    std_time = time.time() - t0

    # Self-RAG
    self_result = self_rag(query, top_k=5)

    print(f"  {'':>18} {'Standard RAG':<25} {'Self-RAG':<25}")
    print(f"  {'Retrieved':>18} {'Always (3 passages)':<25} "
          f"{self_result['retrieve_decision'] + f' ({self_result["relevant_passages"]} relevant)':<25}")
    print(f"  {'Support':>18} {'(not checked)':<25} {self_result['support']:<25}")
    print(f"  {'Utility':>18} {'(not checked)':<25} {str(self_result['utility'])+'/5':<25}")
    print(f"  {'Time':>18} {f'{std_time:.1f}s':<25} {f'{self_result["elapsed_seconds"]:.1f}s':<25}")
    print(f"  {'Attempts':>18} {'1':<25} {str(self_result['attempts']):<25}")
    print(f"\n  Standard answer: {std_result['answer'][:100]}...")
    print(f"  Self-RAG answer: {self_result['answer'][:100]}...")

### Comparison Analysis

| Aspect | Standard RAG | Self-RAG |
|---|---|---|
| **Retrieval** | Always retrieves, even for math/creative tasks | Skips retrieval when not needed |
| **Relevance** | Uses ALL retrieved passages blindly | Filters to only relevant passages |
| **Verification** | No support checking | Verifies answer against evidence |
| **Quality** | No quality signal | Rates and can retry if quality is low |
| **Latency** | 1 retrieval + 1 generation | 4-6 LLM calls per query |
| **Best for** | Simple, uniform question-answering | High-stakes, factual accuracy critical |

The key insight: Self-RAG trades **latency** for **accuracy**. Each reflection step adds
an LLM call, but the resulting answers are more reliable, better grounded, and less prone
to hallucination.

## 13 — Latency Analysis

Let's measure the actual cost of Self-RAG's reflection steps.

In [ ]:
# Time each reflection step independently
import time

query = "What is the levelized cost of solar electricity?"

print("Timing individual reflection steps:")
print("=" * 50)

# Time retrieval decision
t0 = time.time()
check_retrieval_need(query)
t_retrieve = time.time() - t0
print(f"  [Retrieve] decision:  {t_retrieve:.2f}s")

# Time passage retrieval (FAISS — nearly instant)
t0 = time.time()
passages = retrieve(query, top_k=5)
t_faiss = time.time() - t0
print(f"  FAISS retrieval:      {t_faiss:.4f}s")

# Time relevance check (per passage)
t0 = time.time()
for p in passages:
    check_relevance(query, p["passage"])
t_relevant = time.time() - t0
print(f"  [Relevant] (5 pass.): {t_relevant:.2f}s  ({t_relevant/5:.2f}s each)")

# Time generation
context = "\n".join(p["passage"] for p in passages[:3])
t0 = time.time()
answer = generate(f"Answer: {query}\nContext: {context}", max_new_tokens=200)
t_generate = time.time() - t0
print(f"  Generation:           {t_generate:.2f}s")

# Time support check
t0 = time.time()
check_support(query, answer, [p["passage"] for p in passages[:3]])
t_support = time.time() - t0
print(f"  [Supported] check:    {t_support:.2f}s")

# Time utility check
t0 = time.time()
check_utility(query, answer)
t_utility = time.time() - t0
print(f"  [Useful] rating:      {t_utility:.2f}s")

total_self = t_retrieve + t_faiss + t_relevant + t_generate + t_support + t_utility
total_std = t_faiss + t_generate
print(f"\n{'─'*50}")
print(f"  Standard RAG total:  ~{total_std:.2f}s  (retrieve + generate)")
print(f"  Self-RAG total:      ~{total_self:.2f}s  (all reflection steps)")
print(f"  Overhead factor:      {total_self/max(total_std, 0.01):.1f}×")

## 14 — When Is Self-RAG Worth It?

### ✅ Use Self-RAG When:
- **Factual accuracy is critical** — medical, legal, financial domains where hallucination
  has real consequences.
- **The knowledge base is heterogeneous** — passages from many domains where relevance
  filtering prevents cross-contamination.
- **Queries are diverse** — some need retrieval, others don't. Adaptive retrieval saves
  latency on easy questions.
- **Users need trustworthy answers** — the support verification provides a built-in
  confidence signal.

### ❌ Skip Self-RAG When:
- **Latency is paramount** — chatbots with <1s response time requirements can't afford
  4-6 LLM calls.
- **All queries need retrieval** — if every question requires document lookup (e.g., a
  customer support bot with a product manual), the `[Retrieve]` check is wasted.
- **The knowledge base is small and focused** — with only 10 highly relevant documents,
  relevance filtering adds overhead without benefit.
- **You have a powerful reranker** — a cross-encoder reranker can replace `[Relevant]`
  more efficiently.

## 15 — Production Simplifications

In production, you can optimize Self-RAG by:

### 1. Parallel Relevance Checking
Instead of checking passages sequentially, batch all relevance prompts and run them in
parallel (or use a single prompt that evaluates all passages at once).

### 2. Cheaper Models for Reflection
Use a smaller, faster model (e.g., a 1B classifier) for `[Retrieve]` and `[Relevant]`
decisions, reserving the large model for generation only.

### 3. Confidence Thresholds on Embeddings
Replace `[Relevant]` with a cosine similarity threshold (e.g., >0.7). This eliminates
an LLM call entirely and is nearly free.

### 4. Caching Reflection Decisions
If the same query pattern appears frequently, cache the `[Retrieve]` decision.

### 5. Async / Streaming
Start generating while relevance checks are still running, and abort if all passages
are deemed irrelevant.

### Demo: Batch Relevance Check (Single LLM Call)

Instead of one LLM call per passage, we can evaluate all passages in a single prompt.

In [ ]:
def batch_check_relevance(query, passages):
    """Check relevance of multiple passages in a single LLM call."""
    passage_list = "\n".join(
        f"Passage {i+1}: {p[:150]}..." for i, p in enumerate(passages)
    )

    prompt = f"""You are a relevance evaluation module. For each passage below, determine
if it is relevant to the query. Respond with one line per passage in the format:
"Passage N: relevant" or "Passage N: irrelevant"

Query: {query}

{passage_list}

Evaluations:"""

    response = generate(prompt, max_new_tokens=100, temperature=0.1)
    results = []
    for i in range(len(passages)):
        if f"passage {i+1}: irrelevant" in response.lower():
            results.append("irrelevant")
        else:
            results.append("relevant")
    return results

# Compare: sequential vs batch
query = "What role does the gut microbiome play in human health?"
passages = retrieve(query, top_k=5)
passage_texts = [r["passage"] for r in passages]

# Sequential
t0 = time.time()
seq_results = [check_relevance(query, p) for p in passage_texts]
t_seq = time.time() - t0

# Batch
t0 = time.time()
batch_results = batch_check_relevance(query, passage_texts)
t_batch = time.time() - t0

print(f"Query: {query}")
print(f"\nSequential ({t_seq:.2f}s) vs Batch ({t_batch:.2f}s):")
print(f"{'─'*60}")
for i, (s, b, p) in enumerate(zip(seq_results, batch_results, passage_texts)):
    match = "✓" if s == b else "✗"
    print(f"  [{match}] Seq={s:<12} Batch={b:<12} {p[:50]}...")
print(f"\nSpeedup: {t_seq/max(t_batch, 0.01):.1f}×")

### Demo: Cosine Similarity Threshold (Zero LLM Calls)

We can replace the `[Relevant]` LLM call entirely by using the FAISS similarity score
with a threshold — passages below the threshold are considered irrelevant.

In [ ]:
def retrieve_with_threshold(query, top_k=5, threshold=0.5):
    """Retrieve passages and filter by cosine similarity threshold."""
    results = retrieve(query, top_k=top_k)
    relevant = [r for r in results if r["score"] >= threshold]
    filtered = [r for r in results if r["score"] < threshold]
    return relevant, filtered

# Test on a few queries
test_queries = [
    "What is the LCOE of solar power?",
    "How do transistors work?",
    "What is intermittent fasting?",
]

for query in test_queries:
    relevant, filtered = retrieve_with_threshold(query, top_k=5, threshold=0.5)
    print(f"Query: {query}")
    print(f"  Relevant ({len(relevant)}):")
    for r in relevant:
        print(f"    score={r['score']:.4f}  {r['passage'][:60]}...")
    print(f"  Filtered out ({len(filtered)}):")
    for r in filtered:
        print(f"    score={r['score']:.4f}  {r['passage'][:60]}...")
    print()

## 16 — Synthesis: Key Takeaways

### The Self-RAG Philosophy
Self-RAG transforms a passive pipeline into an **active reasoning loop**. The LLM doesn't
just generate — it **reflects** on every decision:

> *"Do I need to look this up?"* → *"Is what I found relevant?"* → *"Is my answer
> actually grounded in the evidence?"* → *"Is this a good answer?"*

### What We Built
1. **Four reflection functions** simulating Self-RAG's special tokens via prompting
2. **An adaptive pipeline** that skips unnecessary retrieval and retries on low quality
3. **Comparison infrastructure** showing concrete improvements over standard RAG
4. **Production optimizations** that reduce the latency overhead

### The Numbers
| Metric | Standard RAG | Self-RAG |
|---|---|---|
| LLM calls per query | 1 | 4–8 |
| Latency | 1× | 2–4× |
| Unnecessary retrieval | Every query | Only when needed |
| Hallucination risk | High | Low (verified) |
| Quality signal | None | Utility 1–5 |

### Connection to Other Techniques
- **CRAG (Corrective RAG)** shares the relevance evaluation step but focuses on web search
  fallback rather than full self-reflection.
- **Adaptive Retrieval** implements just the `[Retrieve]` decision without the other
  reflection tokens.
- **Reliable RAG** adds verification but through external checks rather than self-reflection.

### Further Reading
- Asai et al. (2023). *Self-RAG: Learning to Retrieve, Generate, and Critique through
  Self-Reflection.* [arXiv:2310.11511](https://arxiv.org/abs/2310.11511)
- The original implementation trains reflection tokens as part of the model vocabulary,
  achieving better calibration than our prompt-based simulation.